# YOLO Object Detection

这个 Notebook 使用 `YOLOv8` 预训练模型展示单阶段目标检测的完整推理流程与架构解读。

内容包括：
- YOLOv8 推理与边界框可视化
- CSP Backbone 特征提取
- PANet FPN 多尺度特征融合
- 解耦检测头（Decoupled Head）
- 关键机制：Anchor-free、多尺度、NMS
- YOLO 发展历史（v1 → v8）

## 1. 环境准备

```bash
pip install ultralytics pillow requests
```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO
from dataclasses import dataclass

plt.style.use('seaborn-v0_8')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
from ultralytics import YOLO

# 加载 YOLOv8n（nano，最轻量）预训练模型，首次运行会自动下载权重
model = YOLO('yolov8n.pt')
model.to(device)
print(model.info())

## 2. 准备测试图像

In [ ]:
def load_image_from_url(url):
    resp = requests.get(url, timeout=10)
    return Image.open(BytesIO(resp.content)).convert('RGB')


# 使用公开测试图像（COCO 风格场景）
image_urls = [
    'https://ultralytics.com/images/bus.jpg',
    'https://ultralytics.com/images/zidane.jpg',
]

test_images = [load_image_from_url(u) for u in image_urls]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, img, url in zip(axes, test_images, image_urls):
    ax.imshow(img)
    ax.set_title(url.split('/')[-1])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. 目标检测推理

In [ ]:
# 对所有测试图像批量推理
results = model.predict(source=test_images, conf=0.25, iou=0.7, verbose=False)

for i, result in enumerate(results):
    boxes  = result.boxes
    names  = result.names
    print(f'\nImage {i+1}:')
    print(f'  检测到 {len(boxes)} 个目标')
    for box in boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()
        print(f'  {names[cls]:12s}  conf={conf:.2f}  box={[round(x,1) for x in xyxy]}')

In [ ]:
# 用 matplotlib 手动绘制检测结果
COLORS = plt.cm.get_cmap('tab20').colors


def visualize_detections(image_pil, result):
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image_pil)

    boxes = result.boxes
    names = result.names

    for box in boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        color = COLORS[cls % len(COLORS)]
        rect  = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 4, f'{names[cls]} {conf:.2f}',
            color='white', fontsize=9, fontweight='bold',
            bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor='none')
        )

    ax.axis('off')
    plt.tight_layout()
    plt.show()


for img, result in zip(test_images, results):
    visualize_detections(img, result)

## 4. YOLOv8 架构解读

YOLOv8 遵循经典三段式检测架构：

```
Input Image (640×640)
       ↓
  Backbone (CSPDarkNet + C2f)
  P3 (80×80) / P4 (40×40) / P5 (20×20)
       ↓
  Neck (PANet FPN)
  多尺度特征双向融合
       ↓
  Head (Decoupled Head × 3)
  分类分支 + 回归分支（独立，无 Anchor）
       ↓
  NMS 后处理
```

**关键改进（相比 YOLOv5）**：
- Anchor-free：直接预测中心点偏移和宽高，无需预设 Anchor
- Decoupled Head：分类和回归用独立分支，减少任务干扰
- C2f 模块：增加梯度流动路径，参数效率更高

In [ ]:
# 打印 YOLOv8n 各模块结构
print('YOLOv8n 网络层结构：')
print(f'{"idx":>4}  {"type":35}  {"params":>10}')
print('-' * 55)
total = 0
for i, m in enumerate(model.model.model):
    n = m.__class__.__name__
    p = sum(x.numel() for x in m.parameters())
    total += p
    print(f'{i:>4}  {n:35}  {p:>10,}')
print('-' * 55)
print(f'Total: {total:,} parameters')

In [ ]:
# 提取 backbone 中间特征图并可视化
import torch

img_tensor = torch.from_numpy(
    np.array(test_images[0].resize((640, 640))).astype(np.float32) / 255.0
).permute(2, 0, 1).unsqueeze(0).to(device)

feature_maps = {}

def hook_fn(name):
    def fn(module, inp, out):
        feature_maps[name] = out.detach().cpu()
    return fn

# 注册 hook 提取 P3/P4/P5 特征
backbone_layers = list(model.model.model)
hooks = [
    backbone_layers[4].register_forward_hook(hook_fn('P3_80x80')),
    backbone_layers[6].register_forward_hook(hook_fn('P4_40x40')),
    backbone_layers[8].register_forward_hook(hook_fn('P5_20x20')),
]

with torch.no_grad():
    _ = model.model(img_tensor)

for h in hooks:
    h.remove()

# 可视化各尺度特征图的前 4 个通道
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row, (name, fmap) in enumerate(feature_maps.items()):
    for col in range(4):
        axes[row, col].imshow(fmap[0, col].numpy(), cmap='viridis')
        axes[row, col].set_title(f'{name} ch{col}', fontsize=8)
        axes[row, col].axis('off')

plt.suptitle('Backbone Multi-scale Feature Maps', fontsize=13)
plt.tight_layout()
plt.show()

## 5. 关键机制解读

### Anchor-free 检测
- YOLOv5 之前：每个 grid cell 预测相对于预设 Anchor 的偏移量。
- YOLOv8：直接预测目标中心点和宽高（Distribution Focal Loss），无需人工设计 Anchor。

### 多尺度检测（FPN + PAN）
- P3（80×80）：检测小目标（语义弱，空间分辨率高）
- P4（40×40）：检测中目标
- P5（20×20）：检测大目标（语义强，空间分辨率低）
- FPN 自顶向下融合高层语义，PAN 自底向上传播低层细节。

### NMS（Non-Maximum Suppression）
- 每个位置都会产生预测框，NMS 根据 IoU 去除重复框，保留最高置信度的框。
- DETR 通过 Set Prediction 天然消除重复，无需 NMS。

### 解耦检测头（Decoupled Head）
- 分类分支：判断框内是什么类
- 回归分支：精确定位框的位置
- 两个任务共享 backbone 特征但使用独立的预测层，减少梯度干扰。

In [ ]:
# 对比不同 YOLOv8 规模的参数量
variants = ['yolov8n', 'yolov8s', 'yolov8m']
for v in variants:
    m = YOLO(f'{v}.pt')
    p = sum(x.numel() for x in m.model.parameters())
    print(f'{v:12s}: {p:>10,} parameters')

## 6. YOLO 发展历史

| 版本 | 年份 | 核心贡献 |
|------|------|----------|
| YOLOv1 | 2016 | 单网络端到端检测，Grid cell + 固定 Anchor |
| YOLOv2 | 2017 | Batch Norm、Anchor Boxes、多尺度训练 |
| YOLOv3 | 2018 | Darknet-53、三尺度检测、独立 Sigmoid 分类 |
| YOLOv4 | 2020 | CSPNet Backbone、PANet Neck、Mosaic 增强 |
| YOLOv5 | 2020 | PyTorch 实现、AutoAnchor、自动超参搜索 |
| YOLOv6 | 2022 | 全面 Anchor-free、重参数化、工业部署优化 |
| YOLOv7 | 2022 | E-ELAN 模块、辅助训练头、模型重参数化 |
| YOLOv8 | 2023 | 解耦头、C2f 模块、统一分类/检测/分割框架 |